<a href="https://colab.research.google.com/github/mathewsabby05-gif/AI_Resume_Analyser/blob/main/AI_Resume_Analyser(git).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok PyPDF2 scikit-learn pandas

In [ ]:
%%writefile app.py
# Import Libraries
!pip install streamlit PyPDF2 scikit-learn
import streamlit as st
import pandas as pd
import re
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Page Config
st.set_page_config(page_title="AI Resume Analyzer", layout="wide")
st.title("AI Resume Analyzer & Job Match System")
st.write("Upload your resume and compare it with a job description.")

# Function : Read PDF
def extract_text_from_pdf(uploaded_file):
    text = ""
    reader = PdfReader(uploaded_file)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + " "
    return text

# Function : Clean Text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

# Function : Match Score
def calculate_match_score(resume_text, jd_text):
    corpus = [resume_text, jd_text]
    tfidf = TfidfVectorizer(stop_words="english")
    matrix = tfidf.fit_transform(corpus)
    score = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]
    return round(score * 100, 2)

# Function : Missing Skills
def extract_keywords(text):
    words = text.split()
    keywords = list(set(words))
    return keywords

def find_missing_skills(resume_text, jd_text):
    resume_words = set(extract_keywords(resume_text))
    jd_words = set(extract_keywords(jd_text))

    common_skip = {
        "and","the","for","with","you","your","are","job","role",
        "have","this","that","from","will","our","team","work"
    }

    missing = jd_words - resume_words
    missing = [word for word in missing if len(word) > 2 and word not in common_skip]

    return missing[:20]

# Function : Suggestion
def generate_suggestions(score):
    if score >= 80:
        return "Excellent match! Your resume strongly aligns with the job description."
    elif score >= 60:
        return "Good match. Add more job-specific keywords to improve your chances."
    elif score >= 40:
        return "Average match. Update your skills and experience to better fit the role."
    else:
        return "Low match. Tailor your resume significantly for this position."

# Sidebar
st.sidebar.header("Upload Resume")
uploaded_resume = st.sidebar.file_uploader(
    "Upload Resume (PDF only)", type=["pdf"]
)

# Job description input
job_description = st.text_area(
    "Paste Job Description",
    height=250,
    placeholder="Paste the job description here..."
)

# Analyse Button
if st.button("Analyse Resume"):

    if uploaded_resume is None:
        st.warning("Please upload your resume.")
    elif job_description.strip() == "":
        st.warning("Please paste the job description.")
    else:
        # Read Resume
        resume_text = extract_text_from_pdf(uploaded_resume)

        # Clean Text
        resume_clean = clean_text(resume_text)
        jd_clean = clean_text(job_description)

        # Match Score
        score = calculate_match_score(resume_clean, jd_clean)

        # Missing Skills
        missing_skills = find_missing_skills(resume_clean, jd_clean)

        # Suggestions
        suggestions = generate_suggestions(score)

        # Display Results
        st.subheader("Results")

        col1, col2 = st.columns(2)

        with col1:
            st.metric("Match Score", f"{score}%")

        with col2:
            if score >= 70:
                st.success("Strong Candidate")
            elif score >= 50:
                st.info("Moderate Match")
            else:
                st.error("Needs Improvement")

        st.subheader("Suggestions")
        st.write(suggestions)

        st.subheader("Missing Skills / Keywords")
        if missing_skills:
            for skill in missing_skills:
                st.write(f"• {skill}")
        else:
            st.write("No major missing skills found.")

        st.subheader("Resume Summary")
        st.write(
            "Experienced candidate with relevant technical and professional skills. "
            "Recommended to tailor achievements and keywords to the job posting."
        )

        st.subheader("ATS Tips")
        st.write("""
        - Use exact keywords from the job description
        - Add measurable achievements
        - Keep formatting simple
        - Include relevant tools and technologies
        - Use action verbs
        """)

In [7]:
!streamlit run app.py &>/content/logs.txt &

from pyngrok import ngrok
ngrok.set_auth_token("3CctYbJ0qUdUo7VefjQvQxLzTBO_5rCFW9L9SNYAGepcDPymu")

# Kill any existing ngrok tunnels
ngrok.kill()

url = ngrok.connect(8501)
print("Open this link:", url)

Open this link: NgrokTunnel: "https://decode-utmost-prowler.ngrok-free.dev" -> "http://localhost:8501"
